In [9]:
# from https://techwasti.com/python-tutorial-writing-python-data-class-to-csv
def dataclass_to_csv_row(csv_writer, row):
    csv_writer.writerow(row.__dict__.values())
def dataclass_to_csv_header(csv_writer, header):
    csv_writer.writerow(header.__annotations__.keys())


In [28]:
import xml.etree.ElementTree as ET
from dataclasses import dataclass
import itertools

@dataclass
class Taxon:
    taxid: int
    name: str
    rank: str

@dataclass
class LineageInfo:
    species: Taxon
    genus: Taxon
    family: Taxon
    order: Taxon
    phylum: Taxon
    kingdom: Taxon
    domain: Taxon

def parse_taxon(taxon: ET.Element):
    """Extract TaxID, rank, scientific name, and human-readable rank from a <Taxon> element."""
    taxid = int(taxon.find("TaxId").text)
    name = str(taxon.find("ScientificName").text)
    rank = str(taxon.find("Rank").text if taxon.find("Rank") is not None else "no rank")
    return Taxon(taxid, name, rank)

def parse_xml(file_path: str, csv_writer):
    """Parse XML file and extract taxonomical information."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    for taxon in root.findall("Taxon"):
        # Extract species-level info
        tax_data = parse_taxon(taxon)
        lineage_elem = taxon.find('LineageEx')
        if lineage_elem:
            #lineage_arr = lineage.findall('Taxon')
            # Ranks we care about
            wanted_ranks = {"superkingdom", "clade", "phylum", "order", "family", "class", "genus"}

            lineage = {rank: [] if rank == "clade" else None for rank in wanted_ranks}  # clade can appear multiple times

            # Extract Taxon entries
            for taxon in lineage_elem.findall("Taxon"):
                rank = taxon.find("Rank").text
                name = taxon.find("ScientificName").text
                if rank in wanted_ranks:
                    # clade can be multiple, so keep them in a list
                    if rank == "clade":
                        lineage[rank].append(name)
                    else:
                        lineage[rank] = name

            csv_writer.writerow([tax_data.taxid, tax_data.name, lineage['superkingdom'], *lineage['clade'], lineage['phylum'], lineage['order'], lineage['family'], lineage['genus']])
""" 
        # Extract lineage info
        lineage = taxon.find("LineageEx")
        if lineage is not None:
            print("\nLineage Information:")
            for ancestor in lineage.findall("Taxon")[-7:]:
                lin_data = parse_taxon(ancestor)
                print(lin_data)
        dataclass_to_csv_row(csv_writer, tax_data) """


' \n        # Extract lineage info\n        lineage = taxon.find("LineageEx")\n        if lineage is not None:\n            print("\nLineage Information:")\n            for ancestor in lineage.findall("Taxon")[-7:]:\n                lin_data = parse_taxon(ancestor)\n                print(lin_data)\n        dataclass_to_csv_row(csv_writer, tax_data) '

In [29]:
import csv

with open('data/tax.csv', 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['tax_id', 'species', 'domain', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus'])
    parse_xml('data/taxonomy_result.xml', csv_writer)


In [63]:
import pandas as pd

tax = pd.read_csv('data/cox2_th/taxonomy.tsv')
tax

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3449784,Bremia sp. IK-2025a,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
1,3443803,Phytophthora sp. FA-2025a,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
2,3416268,Phytophthora sp. AUS-9H,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
3,3416267,Phytophthora sp. AUS-9G,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3416266,Phytophthora sp. AUS-9F,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3473,4779,Bremia lactucae,NaN,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3474,4770,Saprolegnia ferax,NaN,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3475,4768,Achlya ambisexualis,NaN,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3476,4767,Achlya klebsiana,NaN,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [11]:
meta = pd.read_csv('data/cox2_th/db-clust.csv', sep='\t')

In [60]:
tax_in_db = tax[tax['tax_id'].isin(meta['tax_id_sanitized'])]
tax_in_db

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
38,3235044,Peronospora wawawaiana,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Peronospora
39,3235043,Peronospora psylliicola,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Peronospora
63,3126754,Phytophthora multiplex,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
68,3090892,Peronosclerospora neglecta,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Peronosclerospora
...,...,...,...,...,...,...,...,...,...
3441,4784,Phytophthora capsici,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
3442,4781,Plasmopara halstedii,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Plasmopara
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia


In [64]:
tax_not_in_db = meta[~meta['tax_id_sanitized'].isin(tax['tax_id'])]
tax_not_in_db

,accession,creation_date,title,authors,sequence_length,organism_name,tax_id,cox2_start,cox2_end,cox2_strand,...,sequence,sequence_length_trimmed,type,cluster_id,query_length,similarity,strand,align_res,query_label,trimed_seq_len


In [32]:
rows_with_nan = tax_in_db[tax_in_db.isna().any(axis=1)]
rows_with_nan

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
171,3033745,Synchrospora medusiformis,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,NaN
489,2584052,Periplasma isogametum,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,NaN,Periplasma
500,2517218,Paralagenidium ajellopsis,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,NaN,Paralagenidium
560,2182428,Crypticola clavulifera,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,NaN,Crypticola
950,1682049,Lagenisma coscinodisci,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Lagenisma
1216,1489789,Lagena radicicola,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,Lagenaceae,Lagena
1323,1440115,Paralagenidium karlingii,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,NaN,Paralagenidium
2341,505981,Haptoglossa zoospora,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Haptoglossa
2741,377434,Haptoglossa heterospora,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Haptoglossa
3364,100866,Leptomitus lacteus,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,NaN,Leptomitus


In [13]:
tax = tax.dropna(subset='family')
tax = tax[~tax['family'].str.startswith('unclassified')]
tax = tax[~tax['family'].str.startswith('environmental')]
tax = tax.dropna(subset='genus')
tax = tax[~tax['genus'].str.startswith('unclassified')]
tax = tax[~tax['genus'].str.startswith('environmental')]
#tax = tax[~tax['order'].str.contains('incertae sedis')]
#tax = tax[~tax['family'].str.contains('incertae sedis')]
tax

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3445,4768,Achlya ambisexualis,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3446,4767,Achlya klebsiana,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [14]:
import re
def sanitize_org_name(org_name: str) -> str:
    org_name_arr = org_name.split(' ')
    to_remove = ['x', 'var.', 'var', 'aff.', 'aff', 'cf.', 'cf']
    for s in to_remove:
        try:
            org_name_arr.remove(s)
        except ValueError:
            pass
    joined = ' '.join(org_name_arr[:2])
    return re.sub(r"[^a-zA-Z\s\-\.]", "", joined)


In [ ]:
#tax['species'] = tax['species'].apply(sanitize_org_name)
#tax = tax[~tax['species'].str.contains('sp.')]
#tax_unique = tax.copy().drop_duplicates(subset=['species'])
#tax_unique

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3445,4768,Achlya ambisexualis,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3446,4767,Achlya klebsiana,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [16]:
tax.to_csv('data/tax-filtered.csv', sep='\t', index=False)

In [3]:
import pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

In [10]:

records = list(SeqIO.parse('data/sh_general_release_dynamic_all_19.02.2025.fasta', 'fasta'))
def parse_record(record: SeqRecord):
    title = record.id.split('|')
    lineage = title[-1].split(';')
    return lineage
    
records = pd.DataFrame([parse_record(record) for record in records])
records_filtered = records[records[1] == 'p__Oomycota']

In [11]:
records_filtered = records_filtered[~records_filtered[6].str.endswith('_sp')]
records_filtered = records_filtered.drop_duplicates(subset=[6])
records_filtered

,0,1,2,3,4,5,6
28,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_tentaculata
29,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_personensis
520,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Pythiales,f__Pythiaceae,g__Globisporangium,s__Globisporangium_echinulatum
1954,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Pseudoperonospora,s__Pseudoperonospora_cubensis
2108,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Peronospora,s__Peronospora_trifolii-minoris
...,...,...,...,...,...,...,...
155224,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_medicaginis
155225,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_uliginosa
155226,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_gibbosa
155227,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_plurivora


In [12]:
records_filtered[3].value_counts()

3
o__Peronosporales    207
o__Pythiales         142
o__Saprolegniales     45
o__Albuginales        13
o__Lagenidiales        4
o__Leptomitales        2
Name: count, dtype: int64

In [4]:
import pandas as pd

df = pd.read_csv('data/taxonomy/NameUsage.tsv', sep='\t', header=0, names=[
    "id", "parent", "status", "rank", "scientificName", "authorship"
])

# Lookup dicts
id_to_name = dict(zip(df["id"], df["scientificName"]))
id_to_parent = dict(zip(df["id"], df["parent"]))
id_to_rank = dict(zip(df["id"], df["rank"]))

# Function to climb lineage
def build_lineage_dict(taxon_id):
    lineage = {}
    while taxon_id in id_to_parent and taxon_id in id_to_name:
        rank = id_to_rank[taxon_id].lower()
        lineage[rank] = id_to_name[taxon_id]
        taxon_id = id_to_parent[taxon_id]
        if taxon_id not in id_to_name and taxon_id not in id_to_parent:
            break
    return lineage

# Build only for species
lineages = []
for _, row in df[df["rank"] == "species"].iterrows():
    lineage_dict = build_lineage_dict(row["id"])
    lineages.append(lineage_dict)

result = pd.DataFrame(lineages)

# Force consistent rank order
rank_order = ["phylum", "class", "order", "family", "genus", "species"]
result = result.reindex(columns=rank_order)
result


,phylum,class,order,family,genus,species
0,Oomycota,NaN,NaN,NaN,Lagenidicopsis,Lagenidicopsis arctica
1,Oomycota,NaN,NaN,NaN,Salispina,Salispina spinosa
2,Oomycota,NaN,NaN,NaN,Salispina,Salispina lobata
3,Oomycota,NaN,NaN,NaN,Salispina,Salispina intermedia
4,Oomycota,NaN,NaN,NaN,Salispina,Salispina hoi
...,...,...,...,...,...,...
1668,Oomycota,Hyphochytrea,Hyphochytriales,Rhizidiomycetaceae,Rhizidiomyces,Rhizidiomyces saprophyticus
1669,Oomycota,Hyphochytrea,Hyphochytriales,Rhizidiomycetaceae,Rhizidiomyces,Rhizidiomyces coronus
1670,Oomycota,Hyphochytrea,Hyphochytriales,Rhizidiomycetaceae,Rhizidiomyces,Rhizidiomyces ichneumon
1671,Oomycota,Hyphochytrea,Hyphochytriales,Rhizidiomycetaceae,Rhizidiomyces,Rhizidiomyces apophysatus


In [36]:
backup = result[result['species'].isin(rows_with_nan['species'])]
backup

,phylum,class,order,family,genus,species
7,Oomycota,NaN,NaN,Lagenaceae,Lagena,Lagena radicicola
46,Oomycota,Peronosporea,Haptoglossales,Haptoglossaceae,Haptoglossa,Haptoglossa heterospora
50,Oomycota,Peronosporea,Haptoglossales,Haptoglossaceae,Haptoglossa,Haptoglossa zoospora
418,Oomycota,Peronosporea,Peronosporales,Crypticolaceae,Crypticola,Crypticola clavulifera
1400,Oomycota,Peronosporea,Peronosporales,Pythiaceae,Paralagenidium,Paralagenidium karlingii
1605,Oomycota,Peronosporea,Leptomitales,Leptomitaceae,Leptomitus,Leptomitus lacteus
1613,Oomycota,Peronosporea,Leptomitales,Leptomitaceae,Apodachlya,Apodachlya pirifera
1621,Oomycota,Peronosporea,Lagenismatales,Lagenismataceae,Lagenisma,Lagenisma coscinodisci


In [46]:
merged = rows_with_nan.merge(result, on='species', suffixes=('_df1', '_backup'), how='left')
merged

,tax_id,species,domain,kingdom,phylum_df1,class_df1,order_df1,family_df1,genus_df1,phylum_backup,class_backup,order_backup,family_backup,genus_backup
0,3033745,Synchrospora medusiformis,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,NaN,NaN,NaN,NaN,NaN,NaN
1,2584052,Periplasma isogametum,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,NaN,Periplasma,NaN,NaN,NaN,NaN,NaN
2,2517218,Paralagenidium ajellopsis,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,NaN,Paralagenidium,NaN,NaN,NaN,NaN,NaN
3,2182428,Crypticola clavulifera,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,NaN,Crypticola,Oomycota,Peronosporea,Peronosporales,Crypticolaceae,Crypticola
4,1682049,Lagenisma coscinodisci,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Lagenisma,Oomycota,Peronosporea,Lagenismatales,Lagenismataceae,Lagenisma
5,1489789,Lagena radicicola,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,Lagenaceae,Lagena,Oomycota,NaN,NaN,Lagenaceae,Lagena
6,1440115,Paralagenidium karlingii,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,NaN,Paralagenidium,Oomycota,Peronosporea,Peronosporales,Pythiaceae,Paralagenidium
7,505981,Haptoglossa zoospora,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Haptoglossa,Oomycota,Peronosporea,Haptoglossales,Haptoglossaceae,Haptoglossa
8,377434,Haptoglossa heterospora,Eukaryota,Sar,Stramenopiles,Oomycota,NaN,NaN,Haptoglossa,Oomycota,Peronosporea,Haptoglossales,Haptoglossaceae,Haptoglossa
9,100866,Leptomitus lacteus,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,NaN,Leptomitus,Oomycota,Peronosporea,Leptomitales,Leptomitaceae,Leptomitus


In [ ]:
import pandas as pd


# The taxonomic ranks
tax_ranks = ['order', 'family', 'genus']

# Function to fill missing values if other ranks match
def fill_taxonomic_blanks(row, df2, ranks):
    # Determine which ranks are missing
    missing_ranks = [r for r in ranks if pd.isna(row[r])]
    
    # For non-missing ranks, filter df2 rows where they match
    filter_mask = pd.Series(True, index=df2.index)
    for r in ranks:
        if r not in missing_ranks:
            filter_mask &= (df2[r] == row[r])
    
    # If a match exists, fill missing ranks
    if filter_mask.any():
        match = df2.loc[filter_mask].iloc[0]
        for r in missing_ranks:
            if r in df2.columns:
                row[r] = match[r]
    
    # For any remaining missing ranks, append 'incertae sedis' to the next higher rank
    ranks = ['class', 'order', 'family', 'genus']
    for i, r in enumerate(ranks):
        if pd.isna(row[r]):
            if r == 'genus':
                # Extract genus from species name (first word)
                row[r] = row['species'].split(' ')[0]
            else:
                # Look for the closest known higher rank
                higher_rank = None
                for j in range(i-1, -1, -1):
                    if pd.notna(row[ranks[j]]):
                        higher_rank = row[ranks[j]]
                        break
                if higher_rank is not None:
                    row[r] = f"{higher_rank} incertae sedis"
                else:
                    row[r] = "incertae sedis"  # fallback if nothing above is known
    return row

df_filled = rows_with_nan.apply(lambda row: fill_taxonomic_blanks(row, result, tax_ranks), axis=1)

df_filled


,tax_id,species,domain,kingdom,phylum,class,order,family,genus
171,3033745,Synchrospora medusiformis,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Synchrospora
489,2584052,Periplasma isogametum,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,Leptomitales incertae sedis,Periplasma
500,2517218,Paralagenidium ajellopsis,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,Lagenidiales incertae sedis,Paralagenidium
560,2182428,Crypticola clavulifera,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Crypticolaceae,Crypticola
950,1682049,Lagenisma coscinodisci,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenismatales,Lagenismataceae,Lagenisma
1216,1489789,Lagena radicicola,Eukaryota,Sar,Stramenopiles,Oomycota,Oomycota incertae sedis,Lagenaceae,Lagena
1323,1440115,Paralagenidium karlingii,Eukaryota,Sar,Stramenopiles,Oomycota,Lagenidiales,Lagenidiales incertae sedis,Paralagenidium
2341,505981,Haptoglossa zoospora,Eukaryota,Sar,Stramenopiles,Oomycota,Haptoglossales,Haptoglossaceae,Haptoglossa
2741,377434,Haptoglossa heterospora,Eukaryota,Sar,Stramenopiles,Oomycota,Haptoglossales,Haptoglossaceae,Haptoglossa
3364,100866,Leptomitus lacteus,Eukaryota,Sar,Stramenopiles,Oomycota,Leptomitales,Leptomitaceae,Leptomitus


,phylum,class,order,family,genus,species
5,Oomycota,NaN,NaN,Lagenaceae,Pythiella,Pythiella vernalis
6,Oomycota,NaN,NaN,Lagenaceae,Ciliatomyces,Ciliatomyces spectabilis
7,Oomycota,NaN,NaN,Lagenaceae,Lagena,Lagena radicicola


In [10]:
e = tax[~tax['species'].isin(result['species'])]
e

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
5,3375942,Phytophthora sp. 'x amnicola-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3382,83918,Pythium sp. Py55,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Pythium
3383,83917,Pythium catenulatum,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Pythium
3384,82955,Pythium sp. Py37,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Pythium
3394,82940,Globisporangium middletonii,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Globisporangium


In [5]:
import pandas as pd
from src import database
from Bio import SeqIO

df = pd.read_csv('data/cox2_th_far_outgroup/seq-meta.csv')
filtered, removed = database.filter_data(df)
filtered.to_csv('data/cox2_th_far_outgroup/seq-meta-filtered.tsv', sep='\t', index=False)
removed.to_csv('data/cox2_th_far_outgroup/seq-meta-filtered-removed.tsv', sep='\t', index=False)
records = SeqIO.parse('data/cox2_th_far_outgroup/sequences.fasta', 'fasta')
database.write_fasta(records, filtered['accession'].values, 'data/cox2_th_far_outgroup/seq-filtered.fasta')
records = SeqIO.parse('data/cox2_th_far_outgroup/sequences.fasta', 'fasta')
database.write_fasta(records, removed['accession'].values, 'data/cox2_th_far_outgroup/seq-filtered-removed.fasta')

In [ ]:
from src import taxonomy

out_files = await taxonomy.download_taxonomy('Spermatophyta', 'data/rbcl/taxonomy/results.xml')

<?xml version="1.0" encoding="UTF-8" ?>
<!DOCTYPE eSearchResult PUBLIC "-//NLM//DTD esearch 20060628//EN" "https://eutils.ncbi.nlm.nih.gov/eutils/dtd/20060628/esearch.dtd">
<eSearchResult><Count>200323</Count><RetMax>20</RetMax><RetStart>0</RetStart><QueryKey>1</QueryKey><WebEnv>MCID_692c643e156a25b6320591b4</WebEnv><IdList>
<Id>3475008</Id>
<Id>3475007</Id>
<Id>3475006</Id>
<Id>3475005</Id>
<Id>3475004</Id>
<Id>3475003</Id>
<Id>3475002</Id>
<Id>3475001</Id>
<Id>3474999</Id>
<Id>3472999</Id>
<Id>3472731</Id>
<Id>3472730</Id>
<Id>3472728</Id>
<Id>3472727</Id>
<Id>3472726</Id>
<Id>3472626</Id>
<Id>3472625</Id>
<Id>3472624</Id>
<Id>3472161</Id>
<Id>3472160</Id>
</IdList><TranslationSet/><TranslationStack>   <TermSet>    <Term>Spermatophyta[Subtree]</Term>    <Field>Subtree</Field>    <Count>239623</Count>    <Explode>N</Explode>   </TermSet>   <TermSet>    <Term>species[Rank]</Term>    <Field>Rank</Field>    <Count>2223663</Count>    <Explode>N</Explode>   </TermSet>   <OP>AND</OP>  </Tra

In [ ]:
import csv

with open('data/rbcl/taxonomy/parsed.csv', 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    taxonomy.write_csv_header(csv_writer)
    for file in out_files:
        with open(file, 'r', encoding='utf-8') as f:
            taxa = taxonomy.parse_xml(f.read())
            taxonomy.to_csv(taxa, csv_writer, header=False)